# NS10 Tutorial A - The Attractiveness Model

**Lecture:** NS10 - Preferential Attachment Extensions

**Reading:** Dorogovtsev, Mendes, and Samukhin (2000)

## Learning goals
- Use the slide rule
  $$
  P(i \leftrightarrow j) = \frac{A + k_j}{\sum_l (A + k_l)}.
  $$
- Understand why a positive baseline attractiveness matters conceptually.
- Simulate the model for different values of $A$ and compare it to BA.
- Quantify how attractiveness changes heterogeneity and first-mover dominance.


In [ ]:
from netsci_utils import *
import pandas as pd

set_seeds()
%matplotlib inline


def attractiveness_model(n, m, A, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    G = nx.complete_graph(m + 1)
    for new_node in range(m + 1, n):
        existing = np.array(list(G.nodes()))
        degrees = np.array([G.degree(node) for node in existing], dtype=float)
        weights = degrees + A
        probs = weights / weights.sum()
        targets = rng.choice(existing, size=m, replace=False, p=probs)
        G.add_node(new_node)
        for target in targets:
            G.add_edge(new_node, int(target))
    return G


def early_node_share(G, fraction=0.05):
    cutoff = max(1, round(fraction * G.number_of_nodes()))
    degrees = dict(G.degree())
    early_degree = sum(degrees[node] for node in range(cutoff))
    total_degree = sum(degrees.values())
    return early_degree / total_degree


def mean_degree_of_recent_nodes(G, fraction=0.25):
    start = int((1 - fraction) * G.number_of_nodes())
    degrees = dict(G.degree())
    return np.mean([degrees[node] for node in range(start, G.number_of_nodes())])

---
## 1. Why add baseline attractiveness?

The slide motivation is sharp: if attachment depends only on current degree, then nodes with very low visibility are penalized too strongly.

This is especially important in citation or web settings, where a node can begin with essentially no incoming attention.


In [ ]:
example_degrees = np.array([0, 1, 5, 20], dtype=float)
node_labels = ['newcomer', 'small', 'medium', 'hub']

rows = []
for A in [0, 2, 6]:
    probs = (example_degrees + A) / np.sum(example_degrees + A)
    for label, degree, prob in zip(node_labels, example_degrees, probs):
        rows.append({'A': A, 'node type': label, 'degree': degree, 'attachment probability': prob})

probability_df = pd.DataFrame(rows)
print(probability_df.round(4).to_string(index=False))


**Interpretation.**
- With $A=0$, the node of degree 0 has zero attachment probability.
- With $A>0$, every node has a baseline chance.
- The model still rewards degree, but it no longer makes visibility entirely dependent on previous success.


---
## 2. Simulate the model for different values of $A$

The slide summary says:
- $A=0$ recovers BA,
- larger $A$ reduces the dominance of degree differences,
- the tail becomes less broad.


In [ ]:
A_values = [0, 2, 6]
attractiveness_graphs = {
    f'A = {A}': attractiveness_model(1500, 2, A=A, seed=RANDOM_SEED)
    for A in A_values
}

summary_rows = []
for A, graph in zip(A_values, attractiveness_graphs.values()):
    row = model_summary_row(graph, f'A = {A}')
    row['early 5% degree share'] = early_node_share(graph, fraction=0.05)
    row['mean degree of latest 25%'] = mean_degree_of_recent_nodes(graph, fraction=0.25)
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
print(summary_df.round(3).to_string(index=False))


In [ ]:
fig, ax = plt.subplots(figsize=FIGURE_SIZE)
for label, color in [
    ('A = 0', CATEGORY_PALETTE['blue']),
    ('A = 2', CATEGORY_PALETTE['orange']),
    ('A = 6', CATEGORY_PALETTE['green']),
]:
    plot_ccdf(
        [degree for _, degree in attractiveness_graphs[label].degree()],
        ax=ax,
        label=label,
        color=color,
        markersize=3,
    )
ax.set_title('Attractiveness smooths but does not remove reinforcement')
ax.legend(frameon=False)
plt.show()


---
## 3. Structural consequence: the tail exponent becomes tunable

The lecture formula is
$$
\gamma = 3 + \frac{A}{m}.
$$

We compare that theoretical trend to a simple empirical tail estimate. The estimate is intentionally rough; it is only used to show the direction of change.


In [ ]:
slope_rows = []
for A, label in zip(A_values, attractiveness_graphs.keys()):
    degrees = [degree for _, degree in attractiveness_graphs[label].degree()]
    slope = naive_ccdf_slope(degrees, kmin=10)
    gamma_hat = 1 - slope if not np.isnan(slope) else np.nan
    slope_rows.append({
        'A': A,
        'theoretical gamma': 3 + A / 2,
        'naive gamma from CCDF tail': gamma_hat,
    })

slope_df = pd.DataFrame(slope_rows)
print(slope_df.round(3).to_string(index=False))


---
## 4. First-mover dominance becomes weaker

If attractiveness really reduces first-mover dominance, then the earliest nodes should capture a smaller share of the total degree when $A$ is large.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(
    [str(A) for A in A_values],
    [early_node_share(attractiveness_graphs[f'A = {A}'], 0.05) for A in A_values],
    color=[CATEGORY_PALETTE['blue'], CATEGORY_PALETTE['orange'], CATEGORY_PALETTE['green']],
)
style_axis(
    axes[0],
    title='Degree share captured by the earliest 5% of nodes',
    xlabel='Attractiveness A',
    ylabel='Share of total degree',
)

axes[1].bar(
    [str(A) for A in A_values],
    [mean_degree_of_recent_nodes(attractiveness_graphs[f'A = {A}'], 0.25) for A in A_values],
    color=[CATEGORY_PALETTE['blue'], CATEGORY_PALETTE['orange'], CATEGORY_PALETTE['green']],
)
style_axis(
    axes[1],
    title='Mean degree of the latest 25% of nodes',
    xlabel='Attractiveness A',
    ylabel='Average degree',
)

plt.show()


## Takeaway

- Attractiveness adds a **baseline visibility** term to preferential attachment.
- It solves the zero-baseline problem and softens extreme first-mover advantage.
- The model keeps hub formation but makes the exponent
  $$
  \gamma = 3 + \frac{A}{m}
  $$
  adjustable.
- This is useful for systems where popularity matters, but raw degree is not the whole story from the start.
